# Entrenamiento ResNet50 - HAR (Human Activity Recognition)

Este notebook entrena **ResNet50** (transfer learning desde ImageNet V2) usando una **GPU T4** en Google Colab.

## Instrucciones
1. Verificar que el runtime sea **GPU**: `Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > T4 GPU`
2. Subir el archivo **`datos_har.zip`** cuando se solicite (contiene `dataset_tr/` y `dataset.csv`)
3. Ejecutar todas las celdas
4. Al finalizar, los resultados se descargan automaticamente

### Optimizaciones GPU activas
- **AMP** (Automatic Mixed Precision): entrena en float16 donde es seguro, 2-3x mas rapido en T4
- **TF32**: precision TensorFloat-32 para matmul y convoluciones
- **torch.compile**: compilacion JIT del modelo (PyTorch 2.x)
- **cudnn.benchmark**: autotuning de kernels CUDA para input fijo
- **pin_memory + non_blocking**: transferencias CPU->GPU asincronas

### Estrategia anti-overfitting (2 fases + progressive unfreezing)
- **Fase 1**: entrenar solo head (backbone congelado, 10 epocas)
- **Fase 2**: fine-tuning progresivo con LR diferencial (backbone=5e-5) + warmup
  - Epocas 1-10: solo layer3 + layer4 + fc descongelados
  - Epoca 11+: backbone completo descongelado
- **Resolucion**: 320x240 (mayor detalle que 224x224)
- **Augmentation**: ColorJitter, RandAugment(mag=6), RandomErasing(0.10)
- **MixUp/CutMix**: desactivado (AUGMIX_PROB=0.0)
- **Regularizacion**: Dropout 0.3, weight_decay 1e-3, label_smoothing 0.05
- **Early stopping**: por test loss con paciencia 10 epocas

## 0. Verificar paquetes y GPU

Instala dependencias faltantes y verifica que la GPU este activa. **No se reinicia el runtime.**

In [ ]:
# Colab ya trae PyTorch+CUDA, numpy, pandas, matplotlib, pillow, scikit-learn.
# Solo instalar lo que pueda faltar sin forzar versiones para no romper Colab.
%pip install -q opencv-python-headless tqdm

# Verificar que todo este disponible
import torch, numpy, pandas, matplotlib, PIL, sklearn, cv2  # noqa: E401
print("Paquetes OK:")
print(f"  torch:        {torch.__version__} (CUDA: {torch.version.cuda})")
print(f"  numpy:        {numpy.__version__}")
print(f"  pandas:       {pandas.__version__}")
print(f"  matplotlib:   {matplotlib.__version__}")
print(f"  pillow:       {PIL.__version__}")
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  opencv:       {cv2.__version__}")
print(f"  CUDA dispo:   {torch.cuda.is_available()}")

In [ ]:
import torch

# Verificar GPU
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU NO DETECTADA. Ir a: Entorno de ejecucion > "
        "Cambiar tipo de entorno de ejecucion > T4 GPU"
    )

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9

# Optimizaciones GPU
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

_test = torch.zeros(1, device=device)
assert _test.is_cuda
del _test

print(f"\nGPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
print(f"TF32: {torch.backends.cuda.matmul.allow_tf32}")
print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")
print("GPU lista.")

## 1. Subir datos

Ejecuta la celda y subi el archivo **`datos_har.zip`**.

In [ ]:
import os
import zipfile
import glob

# Primero buscar si dataset_tr/ ya existe en /content/ (por subida previa)
DATA_DIR = None
for root, dirs, _ in os.walk('/content/'):
    if 'dataset_tr' in dirs:
        DATA_DIR = root
        print(f"Datos ya encontrados en: {root}")
        break

# Si no se encontro, buscar si hay un zip ya subido
if DATA_DIR is None:
    zips = glob.glob('/content/*.zip') + glob.glob('/content/**/*.zip', recursive=True)
    if zips:
        print(f"ZIP encontrado: {zips[0]}, extrayendo...")
        with zipfile.ZipFile(zips[0], 'r') as z:
            z.extractall('/content/')
    else:
        # Subir el zip
        from google.colab import files  # type: ignore[reportMissingImports]
        print("Subi el archivo datos_har.zip ...")
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
        with zipfile.ZipFile(zip_name, 'r') as z:
            z.extractall('/content/')
        os.remove(zip_name)

    # Buscar dataset_tr/ despues de extraer
    for root, dirs, _ in os.walk('/content/'):
        if 'dataset_tr' in dirs:
            DATA_DIR = root
            break

if DATA_DIR is None:
    # Mostrar todo lo que hay para diagnosticar
    print("No se encontro dataset_tr/. Contenido de /content/:")
    for root, dirs, files_list in os.walk('/content/'):
        level = root.replace('/content/', '').count('/')
        if level < 3:  # Solo 3 niveles de profundidad
            indent = '  ' * level
            print(f"{indent}{os.path.basename(root)}/")
            for f in files_list[:5]:
                print(f"{indent}  {f}")
            if len(files_list) > 5:
                print(f"{indent}  ... ({len(files_list)} archivos)")
    raise FileNotFoundError("No se encontro dataset_tr/ en ningun nivel")

IMG_DIR = os.path.join(DATA_DIR, 'dataset_tr')
CSV_PATH = os.path.join(DATA_DIR, 'dataset.csv')

n_imgs = len(os.listdir(IMG_DIR))
print(f"\nDATA_DIR: {DATA_DIR}")
print(f"Imagenes en dataset_tr/: {n_imgs}")
print(f"CSV: {CSV_PATH} ({'existe' if os.path.isfile(CSV_PATH) else 'NO ENCONTRADO'})")

## 2. Imports, configuracion y AMP

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler  # AMP (Mixed Precision)
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# ── Rutas ──
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hiperparametros ──
SEED = 42
BATCH_SIZE = 128          # Mayor batch en GPU (T4 tiene 15 GB VRAM)
PHASE1_EPOCHS = 10        # Fase 1: solo head (backbone congelado)
PHASE2_EPOCHS = 100       # Fase 2: fine-tuning completo (early stopping puede cortar)
LR = 1e-3                 # Learning rate del head (fase 1)
LR_HEAD_PHASE2 = 1e-4     # Learning rate del head (fase 2, mas bajo para no desestabilizar)
LR_BACKBONE = 5e-5        # Learning rate del backbone (fase 2, ajuste mas agresivo)
WEIGHT_DECAY = 1e-3       # Regularizacion L2
IMG_H, IMG_W = 320, 240   # Resolucion mayor para capturar mas detalle
NUM_WORKERS = 2
EARLY_STOP_PATIENCE = 10  # Epocas sin mejora (fase 2)
WARMUP_EPOCHS = 5         # Warmup lineal (fase 2)
UNFREEZE_AFTER = 10       # Epocas F2 solo layer3+layer4, luego descongelar todo
LABEL_SMOOTHING = 0.05    # Suavizado de etiquetas (bajo para 15 clases)
MIXUP_ALPHA = 0.2         # Alpha para MixUp
CUTMIX_ALPHA = 1.0        # Alpha para CutMix
AUGMIX_PROB = 0.0         # MixUp/CutMix desactivado (no ayuda con 15 clases y 12K imgs)
USE_AMP = True            # Automatic Mixed Precision (float16)

# Normalizacion ImageNet (para ResNet50 preentrenado)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

torch.manual_seed(SEED)
np.random.seed(SEED)

assert torch.cuda.is_available(), "GPU no disponible, revisar runtime de Colab"
print(f"Device: {device} ({torch.cuda.get_device_name(0)})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Input: {IMG_H}x{IMG_W} (resolucion aumentada)")
print(f"AMP (Mixed Precision): {USE_AMP}")
print(f"Entrenamiento: 2 fases (F1={PHASE1_EPOCHS} + F2={PHASE2_EPOCHS} epocas)")
print(f"Progressive unfreezing: layer3+layer4 primero, todo tras epoca {UNFREEZE_AFTER}")
print("Configuracion lista.")

## 3. Carga de datos y split train/test

In [ ]:
df_all = pd.read_csv(CSV_PATH)

# Quedarse solo con las imagenes que existen en dataset_tr
existing_files = set(os.listdir(IMG_DIR))
df = df_all[df_all["filename"].isin(existing_files)].copy()
df.reset_index(drop=True, inplace=True)

# Codificar etiquetas
labels_sorted = sorted(df["label"].unique())
label2idx = {label: idx for idx, label in enumerate(labels_sorted)}
idx2label = {idx: label for label, idx in label2idx.items()}
df["label_idx"] = df["label"].map(label2idx)

NUM_CLASSES = len(labels_sorted)
print(f"Imagenes encontradas: {len(df)}")
print(f"Clases: {NUM_CLASSES} -> {labels_sorted}")

# Split
X_train_df, X_test_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["label_idx"]
)
X_train_df.reset_index(drop=True, inplace=True)
X_test_df.reset_index(drop=True, inplace=True)

print(f"Train: {len(X_train_df)} imagenes ({len(X_train_df)/len(df)*100:.0f}%)")
print(f"Test:  {len(X_test_df)} imagenes ({len(X_test_df)/len(df)*100:.0f}%)")

## 4. Modelo ResNet50 (Transfer Learning) + Augmentation

In [ ]:
# ── Data augmentation agresiva (solo train) ──
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_H, IMG_W)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandAugment(num_ops=2, magnitude=6),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.10),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class HARDataset(Dataset):
    """Dataset para imagenes HAR con transformaciones opcionales."""

    def __init__(self, dataframe, img_dir, transform=None):
        self.df = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["filename"])
        img = np.array(Image.open(img_path).convert("RGB"))
        if self.transform:
            img = self.transform(img)
        else:
            pil_img = Image.fromarray(img).resize(
                (IMG_W, IMG_H), Image.Resampling.LANCZOS
            )
            img = np.array(pil_img, dtype=np.float32) / 255.0
            img = np.transpose(img, (2, 0, 1))  # HWC -> CHW
            img = torch.from_numpy(img)
        label = row["label_idx"]
        return img, torch.tensor(label, dtype=torch.long)


train_ds = HARDataset(X_train_df, IMG_DIR, transform=train_transform)
test_ds = HARDataset(X_test_df, IMG_DIR, transform=test_transform)

# Nota: en Colab /dev/shm es pequeño, persistent_workers y prefetch_factor
# pueden causar que los workers mueran por falta de shared memory.
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"DataLoaders creados (RGB, 3 canales):")
print(f"  Input: {IMG_H}x{IMG_W}, Normalize(ImageNet)")
print(f"  Augmentation: ColorJitter + RandAugment(mag=6) + RandomErasing(0.10)")
print(f"  batch={BATCH_SIZE}, workers={NUM_WORKERS}")
print(f"  pin_memory=True")

In [ ]:
# Cargar ResNet50 preentrenado en ImageNet (pesos V2: receta moderna, mejor accuracy)
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

# Reemplazar clasificador final con head custom
model.fc = nn.Sequential(
    nn.Linear(2048, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.3),
    nn.Linear(256, NUM_CLASSES),
)

model = model.to(device)

# Verificar que el modelo esta en GPU
assert next(model.parameters()).is_cuda, "El modelo NO esta en GPU"

# torch.compile: compilacion JIT para optimizar grafos de operaciones (PyTorch 2.x)
try:
    model = torch.compile(model, mode="reduce-overhead")
    compiled = True
except Exception:
    compiled = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Device: {device} ({torch.cuda.get_device_name(0)})")
print(f"Backbone: ResNet50 (preentrenado ImageNet V2)")
print(f"torch.compile: {'activo (reduce-overhead)' if compiled else 'no disponible'}")
print(f"Parametros totales:     {total_params:,}")
print(f"Parametros entrenables: {trainable_params:,}")
print(f"Head: Linear(2048->256) -> ReLU -> Dropout(0.3) -> Linear(256->{NUM_CLASSES})")
print(f"Input: 3 x {IMG_H} x {IMG_W} (RGB)")
print(f"Output: {NUM_CLASSES} clases")

## 5. Entrenamiento en 2 fases

In [ ]:
import time as _time


# ── Helpers: MixUp y CutMix ──
def mixup_data(x, y, alpha=0.2):
    """MixUp: mezcla pares de imagenes y etiquetas."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam


def cutmix_data(x, y, alpha=1.0):
    """CutMix: recorta y pega regiones entre imagenes."""
    lam = np.random.beta(alpha, alpha)
    index = torch.randperm(x.size(0), device=x.device)
    _, _, h, w = x.shape
    cut_rat = np.sqrt(1.0 - lam)
    cut_w, cut_h = int(w * cut_rat), int(h * cut_rat)
    cx, cy = np.random.randint(w), np.random.randint(h)
    bbx1 = np.clip(cx - cut_w // 2, 0, w)
    bby1 = np.clip(cy - cut_h // 2, 0, h)
    bbx2 = np.clip(cx + cut_w // 2, 0, w)
    bby2 = np.clip(cy + cut_h // 2, 0, h)
    mixed_x = x.clone()
    mixed_x[:, :, bby1:bby2, bbx1:bbx2] = x[index, :, bby1:bby2, bbx1:bbx2]
    lam = 1 - (bbx2 - bbx1) * (bby2 - bby1) / (w * h)
    return mixed_x, y, y[index], lam


def _build_optimizer_and_scheduler(mdl):
    """Crea optimizer con LR diferencial y scheduler."""
    backbone_p = [p for p in mdl.parameters()
                  if id(p) not in _fc_ids and p.requires_grad]
    opt = optim.AdamW([
        {"params": backbone_p, "lr": LR_BACKBONE},
        {"params": list(mdl.fc.parameters()), "lr": LR_HEAD_PHASE2},
    ], weight_decay=WEIGHT_DECAY)
    warmup = optim.lr_scheduler.LinearLR(
        opt, start_factor=0.1, end_factor=1.0,
        total_iters=WARMUP_EPOCHS)
    cosine = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=PHASE2_EPOCHS - WARMUP_EPOCHS, eta_min=1e-6)
    sched = optim.lr_scheduler.SequentialLR(
        opt, [warmup, cosine], milestones=[WARMUP_EPOCHS])
    return opt, sched


_t0 = _time.time()

criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
scaler = GradScaler("cuda", enabled=USE_AMP)

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
best_test_acc = 0.0
best_test_loss = float("inf")

print(f"Label smoothing: {LABEL_SMOOTHING}  |  Gradient clipping: 1.0")
print(f"AMP (Mixed Precision): {USE_AMP}")
print("-" * 80)

# ══════════════════════════════════════════════════════════
# FASE 1: Entrenar solo el head (backbone congelado)
# ══════════════════════════════════════════════════════════
# Identificar parametros del head por identidad de objeto (id), no por nombre.
# torch.compile altera los prefijos de nombre de forma impredecible entre
# versiones de PyTorch, asi que comparar nombres no es confiable.
_fc_ids = {id(p) for p in model.fc.parameters()}

for param in model.parameters():
    if id(param) not in _fc_ids:
        param.requires_grad = False

optimizer1 = optim.AdamW(model.fc.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
trainable1 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n--- FASE 1: Entrenar head ({PHASE1_EPOCHS} epocas, backbone congelado) ---")
print(f"Parametros entrenables: {trainable1:,}")

for epoch in range(1, PHASE1_EPOCHS + 1):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer1.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=USE_AMP):
            outputs = model(imgs)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer1)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer1)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad(), autocast("cuda", enabled=USE_AMP):
        for imgs, labels in test_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    test_loss = running_loss / total
    test_acc = correct / total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    if test_loss < best_test_loss:
        best_test_acc = test_acc
        best_test_loss = test_loss
        torch.save(model.state_dict(), OUTPUT_DIR / "har_cnn_best.pth")

    print(f"[F1] Epoca {epoch:03d}/{PHASE1_EPOCHS}  "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f}  |  "
          f"Test Loss: {test_loss:.4f}  Acc: {test_acc:.4f}")

# ══════════════════════════════════════════════════════════
# FASE 2: Fine-tuning progresivo con MixUp/CutMix
# ══════════════════════════════════════════════════════════
# Primeras UNFREEZE_AFTER epocas: solo layer3 + layer4 + fc
# Despues: descongelar todo el backbone

# Detectar layer3/layer4 por identidad de objeto
_layer3_ids = {id(p) for p in model.layer3.parameters()}
_layer4_ids = {id(p) for p in model.layer4.parameters()}
_partial_ids = _fc_ids | _layer3_ids | _layer4_ids

for param in model.parameters():
    param.requires_grad = id(param) in _partial_ids

optimizer2, scheduler2 = _build_optimizer_and_scheduler(model)

trainable2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n--- FASE 2: Fine-tuning progresivo ({PHASE2_EPOCHS} epocas) ---")
print(f"LR head={LR_HEAD_PHASE2}  |  LR backbone={LR_BACKBONE}")
print(f"Parametros entrenables (parcial): {trainable2:,}")
print(f"Descongelar todo tras epoca {UNFREEZE_AFTER}")
print(f"Warmup: {WARMUP_EPOCHS} epocas  |  Early stopping: {EARLY_STOP_PATIENCE} epocas")
print(f"MixUp (alpha={MIXUP_ALPHA}) / CutMix (alpha={CUTMIX_ALPHA}) | prob={AUGMIX_PROB}")

epochs_no_improve = 0
phase2_trained = 0

for epoch in range(1, PHASE2_EPOCHS + 1):
    global_epoch = PHASE1_EPOCHS + epoch
    phase2_trained = epoch

    # Progressive unfreezing: descongelar todo tras UNFREEZE_AFTER
    if epoch == UNFREEZE_AFTER + 1:
        for param in model.parameters():
            param.requires_grad = True
        optimizer2, scheduler2 = _build_optimizer_and_scheduler(model)
        # Avanzar scheduler las epocas ya transcurridas
        for _ in range(epoch - 1):
            scheduler2.step()
        n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f">>> Descongelando backbone completo (params entrenables: {n_train:,})")

    # --- Train con MixUp/CutMix ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Aplicar MixUp o CutMix con probabilidad AUGMIX_PROB
        if np.random.rand() < AUGMIX_PROB:
            if np.random.rand() < 0.5:
                imgs, targets_a, targets_b, mix_lam = mixup_data(imgs, labels, MIXUP_ALPHA)
            else:
                imgs, targets_a, targets_b, mix_lam = cutmix_data(imgs, labels, CUTMIX_ALPHA)
        else:
            targets_a, targets_b, mix_lam = labels, labels, 1.0

        optimizer2.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=USE_AMP):
            outputs = model(imgs)
            loss = mix_lam * criterion(outputs, targets_a) + (1 - mix_lam) * criterion(outputs, targets_b)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer2)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer2)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        _, preds = outputs.max(1)
        correct += (mix_lam * (preds == targets_a).float()
                    + (1 - mix_lam) * (preds == targets_b).float()).sum().item()
        total += labels.size(0)
    train_loss = running_loss / total
    train_acc = correct / total

    # --- Eval (sin MixUp/CutMix) ---
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad(), autocast("cuda", enabled=USE_AMP):
        for imgs, labels in test_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    test_loss = running_loss / total
    test_acc = correct / total

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    scheduler2.step()

    # Guardar mejor modelo (por test loss)
    if test_loss < best_test_loss:
        best_test_acc = test_acc
        best_test_loss = test_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), OUTPUT_DIR / "har_cnn_best.pth")
    else:
        epochs_no_improve += 1

    elapsed = _time.time() - _t0
    print(f"[F2] Epoca {epoch:03d}/{PHASE2_EPOCHS} (global {global_epoch:03d})  "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f}  |  "
          f"Test Loss: {test_loss:.4f}  Acc: {test_acc:.4f}  |  "
          f"LR: {optimizer2.param_groups[1]['lr']:.1e}  "
          f"[{elapsed/60:.1f} min]")

    # Early stopping
    if epochs_no_improve >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping en epoca {epoch} de fase 2 "
              f"(sin mejora en {EARLY_STOP_PATIENCE} epocas)")
        print(f"Mejor Test Acc: {best_test_acc:.4f}")
        break

total_epochs = PHASE1_EPOCHS + phase2_trained
elapsed_total = _time.time() - _t0

print(f"\nEntrenamiento finalizado. Epocas totales: {total_epochs}")
print(f"Tiempo total: {elapsed_total/60:.1f} min")
print(f"Mejor Test Acc: {best_test_acc:.4f} | Mejor Test Loss: {best_test_loss:.4f}")

## 6. Evaluacion final

In [ ]:
# Cargar el mejor modelo
model.load_state_dict(torch.load(OUTPUT_DIR / "har_cnn_best.pth", weights_only=True))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad(), autocast("cuda", enabled=USE_AMP):
    for imgs, labels in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        outputs = model(imgs)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
cm = confusion_matrix(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=labels_sorted)

print(f"Accuracy global: {acc:.4f} ({acc*100:.2f}%)")
print(f"\nClassification Report:\n")
print(report)

## 7. Graficos

In [ ]:
# Curvas de loss y accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_range, history["train_loss"], label="Train Loss", marker="o", markersize=3)
ax1.plot(epochs_range, history["test_loss"], label="Test Loss", marker="o", markersize=3)
ax1.set_xlabel("Epoca")
ax1.set_ylabel("Loss")
ax1.set_title("Loss por Epoca")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history["train_acc"], label="Train Acc", marker="o", markersize=3)
ax2.plot(epochs_range, history["test_acc"], label="Test Acc", marker="o", markersize=3)
ax2.set_xlabel("Epoca")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy por Epoca")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "curvas_entrenamiento.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado: {OUTPUT_DIR / 'curvas_entrenamiento.png'}")

In [ ]:
# Matriz de confusion
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
ax.set_title("Matriz de Confusion", fontsize=14, fontweight="bold")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

tick_marks = np.arange(NUM_CLASSES)
ax.set_xticks(tick_marks)
ax.set_xticklabels(labels_sorted, rotation=45, ha="right", fontsize=8)
ax.set_yticks(tick_marks)
ax.set_yticklabels(labels_sorted, fontsize=8)
ax.set_xlabel("Prediccion", fontsize=12)
ax.set_ylabel("Real", fontsize=12)

thresh = cm.max() / 2.0
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, str(cm[i, j]),
                ha="center", va="center", fontsize=7,
                color="white" if cm[i, j] > thresh else "black")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "matriz_confusion.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado: {OUTPUT_DIR / 'matriz_confusion.png'}")

## 8. Guardar modelo y metricas

In [ ]:
# Guardar modelo ultimo
torch.save(model.state_dict(), OUTPUT_DIR / "har_cnn.pth")
print(f"Modelo ultimo guardado: {OUTPUT_DIR / 'har_cnn.pth'}")
print(f"Mejor modelo (loss={best_test_loss:.4f}, acc={best_test_acc:.4f}): {OUTPUT_DIR / 'har_cnn_best.pth'}")

# Guardar metricas
metrics = {
    "architecture": "resnet50",
    "transfer_learning": True,
    "pretrained_weights": "IMAGENET1K_V2",
    "training_strategy": "two_phase",
    "accuracy": float(acc),
    "best_test_acc": float(best_test_acc),
    "best_test_loss": float(best_test_loss),
    "phase1_epochs": PHASE1_EPOCHS,
    "phase2_epochs": phase2_trained,
    "total_epochs": total_epochs,
    "batch_size": BATCH_SIZE,
    "learning_rate_head_phase1": LR,
    "learning_rate_head_phase2": LR_HEAD_PHASE2,
    "learning_rate_backbone": LR_BACKBONE,
    "weight_decay": WEIGHT_DECAY,
    "label_smoothing": LABEL_SMOOTHING,
    "gradient_clipping": 1.0,
    "dropout": 0.3,
    "augmix_prob": AUGMIX_PROB,
    "warmup_epochs": WARMUP_EPOCHS,
    "mixup_alpha": MIXUP_ALPHA,
    "cutmix_alpha": CUTMIX_ALPHA,
    "img_h": IMG_H,
    "img_w": IMG_W,
    "train_size": len(X_train_df),
    "test_size": len(X_test_df),
    "num_classes": NUM_CLASSES,
    "classes": labels_sorted,
    "confusion_matrix": cm.tolist(),
    "history": history,
    "classification_report": report,
    "device": str(device),
    "gpu_name": torch.cuda.get_device_name(0),
    "total_params": total_params,
    "optimizations": {
        "amp": USE_AMP,
        "tf32": torch.backends.cuda.matmul.allow_tf32,
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "torch_compile": compiled,
        "pin_memory": True,
    },
}
with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f"Metricas guardadas: {OUTPUT_DIR / 'metrics.json'}")

## 9. Descargar resultados

Se descarga un ZIP con los modelos (.pth), metricas (.json) y graficos (.png).

Despues de descargarlo, extrae el contenido en `data_train/output/` de tu proyecto local.

In [ ]:
import shutil
from google.colab import files  # type: ignore[reportMissingImports]

# Crear zip con los resultados
shutil.make_archive('/content/output_results', 'zip', str(OUTPUT_DIR))

print("Contenido del zip:")
for f in sorted(OUTPUT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.2f} MB)")

# Descargar
files.download('/content/output_results.zip')
print("\nDescarga iniciada. Extraer en data_train/output/ de tu proyecto local.")

## Resumen

In [ ]:
print("=" * 60)
print("RESUMEN — ResNet50 Transfer Learning (2 fases)")
print("=" * 60)
print(f"  GPU:                    {torch.cuda.get_device_name(0)}")
print(f"  Arquitectura:           ResNet50 (ImageNet V2 pretrained)")
print(f"  Parametros totales:     {total_params:,}")
print(f"  Entrada:                3x{IMG_H}x{IMG_W}")
print(f"  Imagenes totales:       {len(df)}")
print(f"  Train / Test:           {len(X_train_df)} / {len(X_test_df)}")
print(f"  Clases:                 {NUM_CLASSES}")
print(f"  Batch size:             {BATCH_SIZE}")
print(f"  Fase 1:                 {PHASE1_EPOCHS} epocas (solo head)")
print(f"  Fase 2:                 {phase2_trained} epocas (fine-tuning)")
print(f"  Epocas totales:         {total_epochs}")
print(f"  Label smoothing:        {LABEL_SMOOTHING}")
print(f"  Gradient clipping:      1.0")
print(f"  Dropout:                0.3")
print(f"  MixUp/CutMix:           prob={AUGMIX_PROB}")
print(f"  Accuracy final (test):  {acc:.4f} ({acc*100:.2f}%)")
print(f"  Mejor Test Acc:         {best_test_acc:.4f} ({best_test_acc*100:.2f}%)")
print(f"  Mejor Test Loss:        {best_test_loss:.4f}")
print(f"  --- Optimizaciones ---")
print(f"  AMP (float16):          {USE_AMP}")
print(f"  TF32:                   {torch.backends.cuda.matmul.allow_tf32}")
print(f"  torch.compile:          {compiled}")
print(f"  cuDNN benchmark:        {torch.backends.cudnn.benchmark}")
print("=" * 60)
print("Entrenamiento completado.")